# Exercise 3.3 — Build a Simple Chatbot with Memory
### *"The program remembers what just happened" — a fancy way of saying it has state*

This is the capstone exercise, and it pulls together everything: strings, loops, dictionaries, conditionals, and functions. You're going to build a command-line Q&A bot that remembers information you tell it earlier in the same conversation and can recall it later — the same basic idea (just far simpler!) behind how every modern AI chat tool keeps track of a conversation.

**Why learn this instead of just using ChatGPT or Claude directly?**

Because this exercise demystifies something that feels like magic: **"memory" in a chatbot is just a dictionary that doesn't get erased between questions.** Once you've built that yourself, you'll understand why AI chat tools can "forget" things once a conversation gets reset, why apps need to explicitly save/load conversation history, and how to think about building your own small custom tools on top of AI APIs later, if you ever want to.

## What you'll practise
- **Dictionaries** — the actual "memory" of our bot
- **Loops** — the conversation loop that keeps asking for input
- **Functions** — organising the bot's behaviour into clean pieces
- **State** — the idea that a program can remember things across multiple steps

## Step 1: The simplest possible "bot" — no memory yet

Let's start with a bot that just answers fixed questions from a lookup dictionary, with no memory at all. This is our starting point.

In [1]:
faq = {
    "what is your name": "I'm SupportBot, a simple demo assistant.",
    "what do you do": "I answer questions and remember what you tell me during our chat.",
    "how do i reset my password": "Go to Settings > Security > Reset Password.",
}

def answer(question):
    question = question.lower().strip().strip("?")
    if question in faq:
        return faq[question]
    return "Sorry, I don't know the answer to that yet."

# quick tests
print(answer("What is your name?"))
print(answer("How do I reset my password"))
print(answer("What's the weather today?"))


I'm SupportBot, a simple demo assistant.
Go to Settings > Security > Reset Password.
Sorry, I don't know the answer to that yet.


This works, but it's not really a *conversation* — every question is answered in total isolation, with zero awareness of anything said earlier. Real conversations aren't like that. If you tell a colleague your name once, you don't expect to repeat it every sentence. Let's fix that.

## Step 2: Give the bot memory using a dictionary

In [2]:
# This dictionary is the bot's "memory" for the current session.
# It starts empty and gets filled in as the conversation goes on.
memory = {}

def remember(key, value):
    memory[key] = value
    print(f"(Got it - I'll remember your {key} is {value}.)")

def recall(key):
    return memory.get(key, None)

# quick test
remember("name", "Davina")
remember("favourite_language", "Python")

print(recall("name"))
print(recall("favourite_language"))
print(recall("favourite_food"))   # we never told it this - should be None


(Got it - I'll remember your name is Davina.)
(Got it - I'll remember your favourite_language is Python.)
Davina
Python
None


**This is the whole trick.** `memory` is just a normal dictionary that lives *outside* any single function call, so it survives between one question and the next. Every time we call `remember()`, we add to it. Every time we call `recall()`, we check what's already there. That persistence — a value that outlives a single function call — is what "state" means in programming.

## Step 3: Detect when the user is telling us something to remember

In [3]:
def try_to_remember(user_input):
    lowered = user_input.lower()

    if "my name is " in lowered:
        # split the ORIGINAL string on the lowercase marker's position,
        # so we keep the name's original capitalisation (e.g. "Davina", not "davina")
        marker_index = lowered.index("my name is ") + len("my name is ")
        name = user_input[marker_index:].strip().strip(".")
        remember("name", name)
        return True

    if "i like " in lowered:
        marker_index = lowered.index("i like ") + len("i like ")
        thing = user_input[marker_index:].strip().strip(".")
        remember("favourite_thing", thing)
        return True

    return False

# quick tests
try_to_remember("My name is Davina")
try_to_remember("I like retro gaming")
print(memory)


(Got it - I'll remember your name is Davina.)
(Got it - I'll remember your favourite_thing is retro gaming.)
{'name': 'Davina', 'favourite_language': 'Python', 'favourite_thing': 'retro gaming'}


## Step 4: Recall memory when asked about it

In [4]:
def try_to_recall(user_input):
    lowered = user_input.lower().strip().strip("?")

    if lowered in ("what is my name", "what's my name", "do you know my name"):
        name = recall("name")
        if name:
            return f"Your name is {name}!"
        else:
            return "You haven't told me your name yet."

    if lowered in ("what do i like", "what's my favourite thing"):
        thing = recall("favourite_thing")
        if thing:
            return f"You told me you like {thing}."
        else:
            return "You haven't told me what you like yet."

    return None  # None means "this function doesn't know how to answer this"

# quick tests
print(try_to_recall("What is my name?"))
print(try_to_recall("What do I like?"))


Your name is Davina!
You told me you like retro gaming.


**Why return `None`** instead of some default text? Because we're about to combine *several* different response strategies (FAQ answers, remembering facts, recalling facts), and each one needs a clean way of saying "not my job, try something else." `None` is Python's standard way of saying "nothing to report here.

## Step 5: Combine everything into one conversation loop

In [5]:
def chatbot_response(user_input):
    # 1. Is the user telling us something to remember?
    if try_to_remember(user_input):
        return "Noted!"

    # 2. Is the user asking us to recall something?
    recalled = try_to_recall(user_input)
    if recalled is not None:
        return recalled

    # 3. Is it a known FAQ question?
    faq_answer = answer(user_input)
    return faq_answer


# Simulate a conversation (since this notebook runs non-interactively,
# we'll feed in a scripted list of messages - see Step 6 for the live version)
conversation = [
    "What is your name?",
    "My name is Davina",
    "What is my name?",
    "I like retro gaming",
    "What do I like?",
    "How do I reset my password",
]

for message in conversation:
    print(f"You: {message}")
    print(f"Bot: {chatbot_response(message)}")
    print()


You: What is your name?
Bot: I'm SupportBot, a simple demo assistant.

You: My name is Davina
(Got it - I'll remember your name is Davina.)
Bot: Noted!

You: What is my name?
Bot: Your name is Davina!

You: I like retro gaming
(Got it - I'll remember your favourite_thing is retro gaming.)
Bot: Noted!

You: What do I like?
Bot: You told me you like retro gaming.

You: How do I reset my password
Bot: Go to Settings > Security > Reset Password.



Look at that conversation: the bot correctly remembers the name and the favourite thing *later* in the conversation, purely because `memory` stayed alive between calls to `chatbot_response()`. Nothing fancy — just a dictionary that doesn't get wiped.

## Step 6: Make it live and interactive

The cell below runs an actual interactive loop — type your own messages! Type `bye` to end the conversation. (If you're reading this notebook without running it live, skip ahead to the wrap-up.)

In [6]:
def run_chatbot():
    print("Bot: Hi! I'm SupportBot. Try telling me your name, or ask me a question. Type 'bye' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in ("bye", "exit", "quit"):
            print("Bot: Goodbye! (And just so you know - my memory of this chat ends here too.)")
            break
        print("Bot:", chatbot_response(user_input))

# Uncomment the line below to actually run it interactively:
# run_chatbot()


## 🎉 You did it

You've built a chatbot with real conversational memory, using nothing but dictionaries, functions, and a loop. More importantly, you now understand *why* it works — and by extension, you understand something real about how every AI chat tool you use handles conversation state, even though those are obviously far more sophisticated under the hood.

---

## 🧪 Try it yourself

1. Add a third thing the bot can remember (e.g. department, or a project name).
2. Add a `"forget everything"` command that clears the `memory` dictionary.
3. **Bonus challenge:** save `memory` to a JSON file with `json.dump()` when the user says "bye", and load it back with `json.load()` when the bot starts — so it remembers you even after you close and reopen the notebook. (This is the difference between "memory within a session" and "memory across sessions" — a distinction that matters a lot in real applications.)

## 💡 Where this goes next
You've now covered strings, loops, dictionaries, conditionals, functions, files, and APIs — the real foundation under almost every "AI-powered" tool you'll encounter. The tools you use going forward, vibe-coded or otherwise, will make a lot more sense now that you've built small versions of their core ideas yourself. Nice work getting all the way through.